In [1]:
!pip install kagglehub spacy pandas openpyxl
!python -m spacy download en_core_web_sm


     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ---- ----------------------------------- 1.3/12.8 MB 9.8 MB/s eta 0:00:02
     ---------- ----------------------------- 3.4/12.8 MB 11.1 MB/s eta 0:00:01
     ------------------- -------------------- 6.3/12.8 MB 11.3 MB/s eta 0:00:01
     ------------------- -------------------- 6.3/12.8 MB 11.3 MB/s eta 0:00:01
     ------------------- -------------------- 6.3/12.8 MB 11.3 MB/s eta 0:00:01
     --------------------------- ------------ 8.9/12.8 MB 7.5 MB/s eta 0:00:01
     ----------------------------------- ---- 11.3/12.8 MB 8.1 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 8.2 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import glob
import kagglehub
import pandas as pd

path = kagglehub.dataset_download("dmytroievdokymov/metallica-songs-w-lyrics")
csv_files = glob.glob(f"{path}/*.csv")
print(csv_files)

df = pd.read_csv(csv_files[0], sep=";")
print(df.shape)
df.head()


['C:\\Users\\MSI\\.cache\\kagglehub\\datasets\\dmytroievdokymov\\metallica-songs-w-lyrics\\versions\\4\\Metallica_songs.csv']
(219, 10)


,Song,Author_band,Songwriters,Duration,Album,Album_type,No_on_album,Release_date,Link,Lyrics
0,2 x 4,Metallica,"James Hetfield,Lars Ulrich,Kirk Hammett",5:28,Load,Album,2.0,1996-06-04,https://www.metallica.com/songs/2x4.html,"I’m gonna make you shake you take you,I’m gonn..."
1,53rd & 3rd,The Ramones,Douglas Colvin,2:21,We're a Happy Family: A Tribute to Ramones,Compilation,4.0,2003-02-11,https://www.metallica.com/songs/53rd-and-3rd.html,"If you think you can well come on man,I was a ..."
2,72 Seasons,Metallica,"James Hetfield,Lars Ulrich,Kirk Hammett",7:39,72 Seasons,Album,1.0,2023-04-14,https://www.metallica.com/songs/72-seasons.html,"Feeding on the wrath of man,Shot down,Traumati..."
3,Ain’t My Bitch,Metallica,"James Hetfield,Lars Ulrich",5:04,Load,Album,1.0,1996-06-04,https://www.metallica.com/songs/aint-my-bitch....,"Outta my way,Outta my day,Out of your mind and..."
4,All Day and All of the Night,The Kinks,Ray Davies,NaN,NaN,NaN,NaN,NaN,https://www.metallica.com/songs/all-day-and-al...,"I’m not content to be with you in the daytime,..."


In [3]:
display(df["Author_band"].value_counts().head(10))
display(df["Album_type"].value_counts())


Author_band
Metallica           147
The Ramones           6
Motörhead             6
Diamond Head          5
The Misfits           5
Black Sabbath         4
Blue Öyster Cult      2
Budgie                2
Soundgarden           2
The Kinks             2
Name: count, dtype: int64

Album_type
Album            120
Single            16
Covers            11
Collaborative     11
Compilation        7
Covers EP          5
EP                 4
Live               2
Name: count, dtype: int64

In [4]:
INCLUDE_COVERS = False

df = df[df["Lyrics"].notna() & (df["Lyrics"].str.strip().str.lower() != "instrumental")]
if not INCLUDE_COVERS:
    df = df[df["Author_band"].str.contains("Metallica", case=False, na=False)]
    df = df[~df["Album_type"].isin(["Covers", "Covers EP"])]
df = df.drop_duplicates(subset=["Song", "Lyrics"])

df["Release Year"] = pd.to_datetime(df["Release_date"], errors="coerce").dt.year.astype("Int64")
df["Album"] = df["Album"].fillna("No album")

print("Songs for analysis:", len(df))

Songs for analysis: 141


In [5]:
import spacy

nlp = spacy.load("en_core_web_sm")
KEEP_POS = {"NOUN", "ADJ", "VERB", "ADV"}
EXTRA_STOP = {"oh", "yeah", "gonna", "wanna", "gotta", "na", "hey", "ooh",
              "ah", "uh", "cause", "em", "ll", "ve"}

rows = []
for _, r in df.iterrows():
    doc = nlp(str(r["Lyrics"]).lower())
    for t in doc:
        if (t.pos_ in KEEP_POS and t.is_alpha and not t.is_stop
                and len(t.lemma_) > 2 and t.lemma_ not in EXTRA_STOP):
            rows.append((t.lemma_, r["Song"], r["Album"], r["Release Year"]))

words = (pd.DataFrame(rows, columns=["Word", "Song Name", "Album", "Release Year"])
           .groupby(["Word", "Song Name", "Album", "Release Year"], dropna=False).size()
           .reset_index(name="Frequency"))

words.head()

,Word,Song Name,Album,Release Year,Frequency
0,abduct,The Judas Kiss,Death Magnetic,2008.0,1
1,abide,Halo On Fire,Hardwired... to Self-Destruct,2016.0,2
2,able,Dragon,Lulu,2011.0,2
3,abnormal,Frustration,Lulu,2011.0,2
4,absolute,One,...And Justice for All,1988.0,1


In [6]:
unique_words = (words.groupby("Word")["Frequency"].sum()
                     .sort_values(ascending=False))
unique_words.to_csv("unique_words.csv")

print("Unique words:", len(unique_words))
unique_words.head(20)

Unique words: 2252


Word
life     125
come     125
away     121
feel     106
time     105
let      101
day       94
hear      92
end       91
way       90
light     88
man       87
want      86
death     80
die       78
fall      77
live      77
kill      76
know      75
eye       75
Name: Frequency, dtype: int64

In [7]:
cat = pd.read_csv("data/category_map.csv")
final = words.merge(cat, on="Word", how="left")
final["Category"] = final["Category"].fillna("Other")
final = final[["Word", "Frequency", "Category", "Song Name", "Album", "Release Year"]]

final.to_excel("metallica_words.xlsx", index=False)
final.head()

,Word,Frequency,Category,Song Name,Album,Release Year
0,abduct,1,Rare words (<3),The Judas Kiss,Death Magnetic,2008.0
1,abide,2,Rare words (<3),Halo On Fire,Hardwired... to Self-Destruct,2016.0
2,able,2,Rare words (<3),Dragon,Lulu,2011.0
3,abnormal,2,Rare words (<3),Frustration,Lulu,2011.0
4,absolute,1,Rare words (<3),One,...And Justice for All,1988.0
